<div style="background:#0A005A;border-radius:18px;padding:28px 32px;color:#fff;font-family:'Reddit Sans', 'Open Sans', system-ui, sans-serif">

<div style="margin-top:22px"><span style="background:#613EFF;color:#fff;padding:5px 12px;border-radius:999px;font-size:.8em;font-weight:700;letter-spacing:.06em">TUTORIAL · NOTEBOOK</span>
<span style="color:#C9C2FF;font-size:.85em;margin-left:8px">Engenharia de Dados · Nível 0 · Fundamentos</span></div>
<h1 style="color:#fff;margin:14px 0 8px;font-size:2.1em;line-height:1.15">Prática: coletar de uma API</h1>
<p style="color:#E4E0FF;margin:0 0 12px;font-size:1.05em">Busque as viagens de uma semana, sobreviva às falhas da API e guarde tudo como veio.</p>
<span style="display:inline-block;background:rgba(255,255,255,.12);color:#fff;padding:5px 12px;border-radius:999px;margin:4px 6px 0 0;font-size:.85em">cerca de 30 minutos</span><span style="display:inline-block;background:rgba(255,255,255,.12);color:#fff;padding:5px 12px;border-radius:999px;margin:4px 6px 0 0;font-size:.85em">Python 3.10+</span><span style="display:inline-block;background:rgba(255,255,255,.12);color:#fff;padding:5px 12px;border-radius:999px;margin:4px 6px 0 0;font-size:.85em">roda no Colab</span><span style="display:inline-block;background:rgba(255,255,255,.12);color:#fff;padding:5px 12px;border-radius:999px;margin:4px 6px 0 0;font-size:.85em">dados fictícios</span>
</div>

## O que você vai fazer

- Percorrer todas as páginas de um dia.
- Tratar três respostas diferentes da API: falha temporária, dia vazio e pedido errado.
- Conferir a coleta contra o total que a própria fonte informa.
- Guardar cada dia como veio, com a hora da coleta.

A explicação completa está na aula: [Prática: coletar de uma API](https://gov-hub.io/capacitacao/doc.html?trilha=engenharia-de-dados&path=docs/tutoriais/engenharia-fundamentos-coleta.md).

## Como usar este notebook

Rode as células **em ordem**, de cima para baixo: clique na célula e aperte <kbd>Shift</kbd> + <kbd>Enter</kbd>. A primeira linha de cada célula diz o que fazer com ela:

| Começa com | O que fazer |
|---|---|
| `# Rode e observe` | Código pronto. Rode, leia a saída e compare com a explicação acima dela. |
| `# Sua vez` | Complete o código onde está `None` e rode. |
| `# Confira` | Rode depois da sua vez: ela mostra `[OK]` se está certo, `[ERRO]` com o que falta, ou `[AVISO]` se você ainda não respondeu. |

<div style="border-left:5px solid #613EFF;background:#F2F1F6;padding:12px 16px;border-radius:10px;margin:10px 0;color:#202020;font-family:'Reddit Sans', 'Open Sans', system-ui, sans-serif"><b style="color:#613EFF">Onde rodar</b><br>No <b>Google Colab</b> (só abrir, sem instalar nada), no <b>VS Code</b> com a extensão Jupyter ou no <b>JupyterLab</b> (<code>pip install jupyterlab</code> e depois <code>jupyter lab</code>). Os dados são fictícios e ficam no próprio notebook: nada vai para fora da sua máquina.</div>

##  Preparação

Instale os pacotes, carregue os quadros de conferência e crie o cenário da Pedala Aurora.

In [ ]:
%pip install -q duckdb httpx pandas

In [ ]:
from IPython.display import HTML, display


def mostrar(tipo, titulo, detalhe=""):
    """Mostra um quadro colorido com o resultado de uma conferência."""
    cor, rotulo = {"ok": ("#613EFF", "[OK]"), "erro": ("#F9006F", "[ERRO]"), "info": ("#0A005A", "[AVISO]")}[tipo]
    display(HTML(
        f'<div style="border-left:5px solid {cor};background:#F2F1F6;padding:10px 14px;'
        f'border-radius:8px;margin:6px 0;color:#202020"><b style="color:{cor}">{rotulo} {titulo}</b>'
        + (f"<br>{detalhe}" if detalhe else "") + "</div>"))


def conferir(resposta, certo, mensagem_certo, mensagem_errado):
    """Compara a sua resposta com a esperada e mostra um quadro verde, rosa ou de aviso."""
    if resposta is None:
        mostrar("info", "Ainda sem resposta", "Complete a célula “Sua vez” acima, rode-a e rode esta de novo.")
    elif certo:
        mostrar("ok", mensagem_certo)
    else:
        mostrar("erro", mensagem_errado)


mostrar("ok", "Quadros de conferência prontos")

A célula abaixo gera os dados e liga uma **API de mentira** aqui mesmo, no notebook. Ela está recolhida porque você não precisa ler o código para seguir; se tiver curiosidade, abra.

In [ ]:
#@title Preparar o cenário (clique duas vezes para ver o código)
# Cria os dados da Pedala Aurora e liga, aqui mesmo no notebook, uma API de mentira que os entrega.
# Os dados são sintéticos e sempre os mesmos (semente fixa). Você não precisa ler este código para seguir.
import json
import random
import threading
from datetime import date, datetime, timedelta, timezone
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import parse_qs, urlparse

FUSO = timezone(timedelta(hours=-3))
ESTACOES = [
    {"id_estacao": "E01", "nome": "Praça Central", "bairro": "Centro", "vagas": 20},
    {"id_estacao": "E02", "nome": "Rodoviária", "bairro": "Centro", "vagas": 15},
    {"id_estacao": "E03", "nome": "Parque das Águas", "bairro": "Jardim Norte", "vagas": 12},
    {"id_estacao": "E04", "nome": "Mercado Municipal", "bairro": "Centro", "vagas": 16},
    {"id_estacao": "E05", "nome": "Universidade", "bairro": "Vila Nova", "vagas": 18},
    {"id_estacao": "E06", "nome": "Hospital Regional", "bairro": "Vila Nova", "vagas": 10},
    {"id_estacao": "E07", "nome": "Orla do Lago", "bairro": "Lagoa", "vagas": 14},
    {"id_estacao": "E08", "nome": "Estação do Metrô", "bairro": "Jardim Sul", "vagas": 20},
]
PRIMEIRO_DIA, ULTIMO_DIA = date(2026, 6, 1), date(2026, 6, 7)
POR_PAGINA = 40


def _gerar():
    sorteio = random.Random(2026)
    pesos = [5, 3, 2, 3, 4, 1, 2, 5]
    horas = list(range(6, 23))
    peso_hora = [2, 6, 9, 5, 3, 3, 4, 3, 3, 3, 4, 7, 9, 6, 3, 2, 1]
    viagens, atrasadas, numero = {}, [], 5000
    dia = PRIMEIRO_DIA
    while dia <= ULTIMO_DIA:
        quantas = sorteio.randint(85, 115) if dia.weekday() < 5 else sorteio.randint(55, 75)
        lista = []
        for _ in range(quantas):
            numero += 1
            saida, chegada = sorteio.choices(ESTACOES, pesos, k=2)
            hora = sorteio.choices(horas, peso_hora)[0]
            inicio = datetime(dia.year, dia.month, dia.day, hora, sorteio.randint(0, 59), tzinfo=FUSO)
            fim = inicio + timedelta(minutes=sorteio.randint(3, 45))
            lista.append({"id_viagem": f"V{numero}", "id_usuario": f"U{sorteio.randint(100, 399)}",
                          "estacao_saida": saida["id_estacao"], "estacao_chegada": chegada["id_estacao"],
                          "inicio": inicio.isoformat(), "fim": fim.isoformat()})
        lista.sort(key=lambda v: v["inicio"])
        viagens[dia.isoformat()] = lista
        dia += timedelta(days=1)
    # Defeitos plantados, como os de uma fonte real:
    for v in viagens["2026-06-04"][10:12]:                 # fim antes do início
        v["inicio"], v["fim"] = v["fim"], v["inicio"]
    for dia_, i in (("2026-06-02", 30), ("2026-06-04", 50), ("2026-06-06", 20)):
        viagens[dia_][i]["estacao_saida"] = "E99"           # estação fora do cadastro
    for _ in range(4):                                      # viagens de 05/06 que só chegam depois
        numero += 1
        inicio = datetime(2026, 6, 5, 21, sorteio.randint(0, 59), tzinfo=FUSO)
        atrasadas.append({"id_viagem": f"V{numero}", "id_usuario": f"U{sorteio.randint(100, 399)}",
                          "estacao_saida": "E07", "estacao_chegada": "E01",
                          "inicio": inicio.isoformat(), "fim": (inicio + timedelta(minutes=12)).isoformat()})
    return viagens, atrasadas


VIAGENS, _ATRASADAS = _gerar()
_estado = {"atrasadas_chegaram": False, "falha_temporaria": True}


def chegam_viagens_atrasadas():
    """Simula o dia seguinte: a fonte recebe 4 viagens de 05/06 que ainda não tinham sido enviadas."""
    if not _estado["atrasadas_chegaram"]:
        VIAGENS["2026-06-05"] = sorted(VIAGENS["2026-06-05"] + _ATRASADAS, key=lambda v: v["inicio"])
        _estado["atrasadas_chegaram"] = True


class _Api(BaseHTTPRequestHandler):
    def log_message(self, *args):
        pass

    def _responder(self, status, corpo=None):
        self.send_response(status)
        if corpo is None:
            self.end_headers()
            return
        dados = json.dumps(corpo, ensure_ascii=False).encode("utf-8")
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(dados)))
        self.end_headers()
        self.wfile.write(dados)

    def do_GET(self):
        url = urlparse(self.path)
        args = {k: v[0] for k, v in parse_qs(url.query).items()}
        if url.path == "/estacoes":
            return self._responder(200, ESTACOES)
        if url.path != "/viagens":
            return self._responder(404, {"erro": "caminho inexistente"})
        try:
            dia = date.fromisoformat(args.get("data", ""))
            pagina = int(args.get("pagina", "1"))
        except ValueError:
            return self._responder(400, {"erro": "use data=AAAA-MM-DD e pagina=1, 2, ..."})
        lista = VIAGENS.get(dia.isoformat(), [])
        if not lista:
            return self._responder(204)
        if dia.isoformat() == "2026-06-02" and pagina == 2 and _estado["falha_temporaria"]:
            _estado["falha_temporaria"] = False
            return self._responder(503, {"erro": "servidor ocupado, tente de novo"})
        paginas = -(-len(lista) // POR_PAGINA)
        inicio = (pagina - 1) * POR_PAGINA
        trecho = lista[inicio:inicio + POR_PAGINA]
        if dia.isoformat() == "2026-06-03" and pagina == 2:
            trecho = [lista[inicio - 1]] + trecho               # a página 2 repete o último da página 1
        self._responder(200, {"data": dia.isoformat(), "pagina": pagina, "total_paginas": paginas,
                              "total_registros": len(lista), "viagens": trecho})


_servidor = ThreadingHTTPServer(("127.0.0.1", 0), _Api)
threading.Thread(target=_servidor.serve_forever, daemon=True).start()
API = f"http://127.0.0.1:{_servidor.server_port}"
print(f"Cenário pronto: {len(ESTACOES)} estações, {sum(len(v) for v in VIAGENS.values())} viagens "
      f"de {PRIMEIRO_DIA:%d/%m} a {ULTIMO_DIA:%d/%m}. API em {API}")

##  Passo 1 de 5 · Pedir todas as páginas de um dia

Duas funções: `pedir` faz um pedido e devolve o JSON; `coletar_dia` chama `pedir` página por página até a última que a API informou.

In [ ]:
# Rode e observe
import httpx


def pedir(caminho, parametros=None):
    resposta = httpx.get(API + caminho, params=parametros, timeout=10)
    resposta.raise_for_status()      # qualquer status de erro vira uma exceção
    return resposta.json()


def coletar_dia(data):
    """Todas as viagens de um dia, página por página, e o total que a API informa."""
    viagens, pagina, total_paginas, total_registros = [], 1, 1, 0
    while pagina <= total_paginas:
        corpo = pedir("/viagens", {"data": data, "pagina": pagina})
        if corpo is None:                    # 204: não há viagens neste dia
            return [], 0
        viagens.extend(corpo["viagens"])
        total_paginas, total_registros = corpo["total_paginas"], corpo["total_registros"]
        pagina += 1
    return viagens, total_registros


viagens, total = coletar_dia("2026-06-01")
print(len(viagens), "viagens coletadas; a API informou", total)

##  Passo 2 de 5 · Quando a API falha

Agora colete 02/06. A API vai falhar no meio do caminho.

In [ ]:
# Rode e observe
try:
    coletar_dia("2026-06-02")
except httpx.HTTPStatusError as erro:
    print("Falhou:", erro.response.status_code, erro.response.json())

<div style="border-left:5px solid #613EFF;background:#F2F1F6;padding:12px 16px;border-radius:10px;margin:10px 0;color:#202020;font-family:'Reddit Sans', 'Open Sans', system-ui, sans-serif"><b style="color:#613EFF">Três respostas, três decisões</b><br><b>503</b> (servidor ocupado) é <b>temporário</b>: vale esperar e tentar de novo. <b>204</b> (sem conteúdo) quer dizer que <b>não há viagens</b> no dia: não é erro, é uma lista vazia. <b>400</b> (pedido errado) é <b>permanente</b>: insistir não resolve, é preciso corrigir o pedido. A nova versão de <code>pedir</code> trata cada caso do seu jeito.</div>

In [ ]:
# Rode e observe
import time

import httpx


def pedir(caminho, parametros=None, tentativas=3):
    """Faz um pedido à API. Tenta de novo quando o erro é do servidor (5xx)."""
    for tentativa in range(1, tentativas + 1):
        resposta = httpx.get(API + caminho, params=parametros, timeout=10)
        if resposta.status_code == 204:
            return None                      # dia sem viagens: a resposta vem vazia
        if resposta.status_code >= 500 and tentativa < tentativas:
            print(f"  {resposta.status_code} na tentativa {tentativa}: nova tentativa em 1 segundo")
            time.sleep(1)
            continue
        resposta.raise_for_status()          # 4xx: o pedido está errado; insistir não resolve
        return resposta.json()


viagens, total = coletar_dia("2026-06-02")
print(len(viagens), "viagens coletadas; a API informou", total)
print("Dia sem viagens:", coletar_dia("2026-06-09"))
try:
    coletar_dia("02/06/2026")
except httpx.HTTPStatusError as erro:
    print("Pedido errado:", erro.response.status_code, erro.response.json()["erro"])

<div style="border-left:5px solid #F9006F;background:#F2F1F6;padding:12px 16px;border-radius:10px;margin:10px 0;color:#202020;font-family:'Reddit Sans', 'Open Sans', system-ui, sans-serif"><b style="color:#F9006F">A falha acontece uma vez</b><br>A API de mentira falha só no primeiro pedido da página 2 de 02/06, como um servidor que estava ocupado naquele instante. Se você rodar as células de novo, ela não falha mais: é assim com falhas temporárias.</div>

##  Passo 3 de 5 · Conferir a coleta

A API informa quantos registros o dia tem. Compare com o que você recebeu e com quantas viagens **diferentes** vieram.

In [ ]:
# Rode e observe
for data in ["2026-06-01", "2026-06-02", "2026-06-03"]:
    viagens, total = coletar_dia(data)
    distintas = len({v["id_viagem"] for v in viagens})
    print(f"{data}: recebidas {len(viagens):>3} · diferentes {distintas:>3} · a API informou {total:>3}")

<div style="border-left:5px solid #613EFF;background:#F2F1F6;padding:12px 16px;border-radius:10px;margin:10px 0;color:#202020;font-family:'Reddit Sans', 'Open Sans', system-ui, sans-serif"><b style="color:#613EFF">Leia assim</b><br>Em 03/06 chegaram 97 linhas para 96 viagens: a página 2 repetiu a última viagem da página 1. Isso acontece em APIs reais quando um registro novo entra enquanto você pagina. A coleta <b>não apaga</b> a repetida: ela guarda o que veio e avisa. A limpeza é trabalho da transformação, mais adiante, e assim a evidência do problema não se perde.</div>

### Sua vez

Escreva `resumo(viagens, total)`: ela devolve o texto `"ok"` quando o número de viagens **diferentes** é igual ao total informado, e `"diferente"` caso contrário. É a conferência que a coleta vai fazer em todo dia.

In [ ]:
# Sua vez
def resumo(viagens, total):
    return None   # troque por "ok" ou "diferente", conforme a conta

In [ ]:
# Confira
testes = [resumo(*coletar_dia("2026-06-03")), resumo([{"id_viagem": "V1"}, {"id_viagem": "V1"}], 2)]
conferir(testes[0], testes == ["ok", "diferente"], "A conferência funciona: ela conta viagens diferentes",
         f"Ainda não: a sua função devolveu {testes}; o esperado é ['ok', 'diferente']")

<details>
<summary><b>Ver uma resposta</b> (tente antes)</summary>

```python
def resumo(viagens, total):
    distintas = len({v["id_viagem"] for v in viagens})
    return "ok" if distintas == total else "diferente"
```
</details>

##  Passo 4 de 5 · Guardar como veio

Cada dia vira um arquivo JSON na pasta `bruto/`, com a hora da coleta. Se algo der errado depois, dá para refazer tudo a partir daqui, sem pedir de novo à API.

In [ ]:
# Rode e observe
import json
from datetime import datetime
from pathlib import Path

PASTA = Path("bruto")


def salvar_bruto(data, viagens, total_registros):
    """Guarda o dia exatamente como veio da API, com a hora da coleta."""
    PASTA.mkdir(exist_ok=True)
    arquivo = PASTA / f"viagens_{data}.json"
    conteudo = {"data": data, "coletado_em": datetime.now().isoformat(timespec="seconds"),
                "total_registros": total_registros, "viagens": viagens}
    arquivo.write_text(json.dumps(conteudo, ensure_ascii=False), encoding="utf-8")
    return arquivo


DIAS = [f"2026-06-0{d}" for d in range(1, 8)]   # de 01/06 a 07/06
for data in DIAS:
    viagens, total = coletar_dia(data)
    arquivo = salvar_bruto(data, viagens, total)
    print(f"{arquivo} · {len(viagens)} viagens")

##  Passo 5 de 5 · Olhar um arquivo

Abra um arquivo para ver o que foi guardado.

In [ ]:
# Rode e observe
conteudo = json.loads((PASTA / "viagens_2026-06-03.json").read_text(encoding="utf-8"))
{chave: (valor if chave != "viagens" else f"{len(valor)} viagens") for chave, valor in conteudo.items()}

## O que você aprendeu

- Uma coleta percorre as páginas até a última que a fonte informa.
- Falha temporária pede nova tentativa; pedido errado pede correção; dia vazio não é erro.
- A conferência compara o que chegou com o que a fonte diz que existe.
- A camada bruta guarda o dado **como veio**, com a hora da coleta.

**Próxima aula:** [Prática: gravar sem duplicar](https://gov-hub.io/capacitacao/doc.html?trilha=engenharia-de-dados&path=docs/tutoriais/engenharia-fundamentos-gravar.md).